### **1.Cleaning Data With Pandas**

In [31]:
import pandas as pd

#### **1.Loading Data**

In [32]:
# import csv file
raw= pd.read_csv('dataset/kakuma_fresh_mart_sales.csv')

In [33]:
# create a copy of the data
clean= raw.copy()
clean.head()

,receipt,date,branch,product,category,qty,price,payment,customer
0,R10001,2026-10-01,Kakuma Town,Water,Drinks,2.0,50,M-Pesa,NaN
1,R10002,2026-10-01,LODWAR,Water,Drinks,4.0,50,mpesa,NaN
2,R10003,2026-10-01,Kakuma Town,Salt,Cooking,2.0,40,Cash,NaN
3,R10004,2026-10-01,Kitale,Milk,Dairy,2.0,65,Cash,C0308
4,R10005,01/10/2026,Kakuma Town,Detergent,Household,3.0,270,Cash,NaN


#### **1.Remove Duplicate**

In [34]:
print('Before Removing duplicates:', clean.shape)
clean = clean.drop_duplicates()
print("After removing duplicates:",clean.shape)

Before Removing duplicates: (2509, 9)
After removing duplicates: (2479, 9)


#### **2.Standardize the Labels**

In [35]:
# 1. Branch
# unique branches
clean['branch'].unique()

<StringArray>
['Kakuma Town',      'LODWAR',      'Kitale',      'Lodwar',      'kitale',
 'kakuma town',      'KITALE', 'KAKUMA TOWN',      'lodwar']
Length: 9, dtype: str

In [36]:
# Remove extra spaces ,convert to title
clean['branch']=clean['branch'].str.strip().str.title()

In [37]:
clean['branch'].unique()

<StringArray>
['Kakuma Town', 'Lodwar', 'Kitale']
Length: 3, dtype: str

In [38]:
#2.Category
clean['category'].unique()

<StringArray>
[   'Drinks',   'Cooking',     'Dairy', 'Household',    'Grains',   'Produce',
   'produce', 'household',    'drinks',     'DAIRY',     'dairy', 'HOUSEHOLD',
   'COOKING',   'cooking',   'PRODUCE',    'DRINKS',    'GRAINS',    'grains']
Length: 18, dtype: str

In [39]:
# remove extra space and convert to lower case
clean['category']=clean['category'].str.strip().str.lower()

In [40]:
clean['category'].unique()

<StringArray>
['drinks', 'cooking', 'dairy', 'household', 'grains', 'produce']
Length: 6, dtype: str

In [41]:
clean['payment'].unique()

<StringArray>
['M-Pesa', 'mpesa', 'Cash', 'Mpesa', 'Card', 'Credit', nan, 'M-PESA']
Length: 8, dtype: str

In [42]:
#remove M-PESA,mpesa,Mpesa to M-pesa

mpesa_map={
    'mpesa':'M-Pesa',
    'Mpesa': 'M-Pesa',
    'M-PESA': 'M-Pesa'

}
clean['payment']= clean['payment'].replace(mpesa_map)

In [43]:
clean['payment'].value_counts(dropna=False)

payment
Cash      1105
M-Pesa    1062
Card       170
Credit      81
NaN         61
Name: count, dtype: int64

In [44]:
clean['payment']= clean['payment'].fillna('unknown')
clean['payment'].value_counts()


payment
Cash       1105
M-Pesa     1062
Card        170
Credit       81
unknown      61
Name: count, dtype: int64

In [45]:
clean['payment'].head()

0    M-Pesa
1    M-Pesa
2      Cash
3      Cash
4      Cash
Name: payment, dtype: str

#### **3.Convert Dates and Types**

In [46]:
# unique dates
clean['date'].unique()

<StringArray>
['2026-10-01', '01/10/2026', '2026-10-02', '02/10/2026', '2026-10-03',
 '03/10/2026', '2026-10-04', '04/10/2026', '2026-10-05', '05/10/2026',
 '2026-10-06', '06/10/2026', '2026-10-07', '07/10/2026', '2026-10-08',
 '08/10/2026', '2026-10-09', '09/10/2026', '2026-10-10', '10/10/2026',
 '2026-10-11', '11/10/2026', '2026-10-12', '12/10/2026', '2026-10-13',
 '13/10/2026', '2026-10-14', '14/10/2026', '15/10/2026', '2026-10-15',
 '2026-10-16', '16/10/2026', '2026-10-17', '17/10/2026', '18/10/2026',
 '2026-10-18', '2026-10-19', '19/10/2026', '2026-10-20', '20/10/2026',
 '2026-10-21', '21/10/2026']
Length: 42, dtype: str

In [47]:
# %Y-%m-%d
iso_dates=pd.to_datetime(clean['date'],format= '%Y-%m-%d',errors= 'coerce')
iso_dates.sample(10)

1468   2026-10-13
2382          NaT
1624   2026-10-14
387    2026-10-04
67     2026-10-01
2175   2026-10-19
271    2026-10-03
243    2026-10-03
1791   2026-10-16
201    2026-10-02
Name: date, dtype: datetime64[us]

In [48]:
# %d/%m/%Y
day_first= pd.to_datetime(clean['date'],format='%d/%m/%Y',errors='coerce')
day_first.sample(10)

1796          NaT
929           NaT
2500          NaT
481           NaT
2274          NaT
1385   2026-10-12
2282          NaT
469           NaT
1611          NaT
1255          NaT
Name: date, dtype: datetime64[us]

In [49]:
# fill missing iso_date with the day_first
clean['date']= iso_dates.fillna(day_first)
clean['date'].sample(20)

454    2026-10-04
542    2026-10-05
2068   2026-10-18
2295   2026-10-20
732    2026-10-07
2007   2026-10-17
1012   2026-10-09
358    2026-10-03
494    2026-10-04
1183   2026-10-10
2242   2026-10-19
1269   2026-10-11
767    2026-10-07
2294   2026-10-20
1809   2026-10-16
450    2026-10-04
1998   2026-10-17
546    2026-10-05
1102   2026-10-10
2145   2026-10-18
Name: date, dtype: datetime64[us]

#### **4. Handle missing and Impossble Values**

In [50]:
#1.missing quantities

#drop missing quantities
clean= clean.dropna(subset='qty')
clean['qty'].unique()

array([  2.,   4.,   3.,   1.,   6.,   7.,   5.,  -1.,   8.,  -2., 999.,
       500., 250.])

In [51]:
# price
clean['price'].unique()

<StringArray>
[ '50',  '40',  '65', '270', '320',  '70', '190',  '85',  '45', '250', '480',
 '120',  '20', '290', '165',  '30',  '27', '180',   nan, '130',  '90', '100',
  '36',  '63',  '58', '288', '162', '108', 'TBC', '261', '148',  '76', '432',
 '171',  '81', '225',  '18', '243', '117',   '0']
Length: 40, dtype: str

In [52]:
clean[clean['price']=='TBC']

,receipt,date,branch,product,category,qty,price,payment,customer
213,R10211,2026-10-02,Kakuma Town,Sugar,cooking,4.0,TBC,M-Pesa,NaN
1240,R11223,2026-10-11,Kakuma Town,Sugar,cooking,3.0,TBC,M-Pesa,C0013
1922,R11898,2026-10-17,Kitale,Soap,household,3.0,TBC,Credit,C0227
2025,R11999,2026-10-18,Kakuma Town,Butter,dairy,1.0,TBC,M-Pesa,NaN


In [53]:
# convert price in to numeric
clean['price']= pd.to_numeric(clean['price'],errors= 'coerce')

clean['price'].unique()

array([ 50.,  40.,  65., 270., 320.,  70., 190.,  85.,  45., 250., 480.,
       120.,  20., 290., 165.,  30.,  27., 180.,  nan, 130.,  90., 100.,
        36.,  63.,  58., 288., 162., 108., 261., 148.,  76., 432., 171.,
        81., 225.,  18., 243., 117.,   0.])

In [54]:
# zero prices
clean[clean['price']==0]

,receipt,date,branch,product,category,qty,price,payment,customer
887,R10874,2026-10-08,Kakuma Town,Soda,drinks,5.0,0.0,M-Pesa,NaN
1174,R11159,2026-10-10,Kakuma Town,Water,drinks,2.0,0.0,Cash,NaN
1874,R11851,2026-10-16,Lodwar,Maize flour,grains,3.0,0.0,Cash,C0139
2048,R12022,2026-10-18,Lodwar,Soda,drinks,5.0,0.0,M-Pesa,NaN
2155,R12128,2026-10-18,Lodwar,Sugar,cooking,2.0,0.0,M-Pesa,NaN


In [55]:
clean.loc[clean['price']==0,'price']=float('nan')

#### **7.flag instead of delete**

In [56]:
# negative
clean['is_return']= clean['qty']<0


# unusual
clean["unusual_qty"]= clean['qty']>100


# missing customers
clean['new_customer']= clean['customer'].isna()

In [57]:
clean['new_customer'] 

0        True
1        True
2        True
3       False
4        True
        ...  
2504    False
2505    False
2506    False
2507     True
2508    False
Name: new_customer, Length: 2467, dtype: bool

In [58]:
# save the dataset
clean.to_csv('dataset/kakuma_fresh_mart_sales_clean.csv',index=False)

print('dataset saved successfully!!')

dataset saved successfully!!
